## In this notebook, we demo the prompting startegy which we will refine for P3 
For this demo, we wil only be working with Deepseek-R1, as it is good and cheap
- This is an example that demonstrats the multi-agent model solving an "easy" category problem using forced exploration.

## Load the selected qustions
For Deepseek-R1 we have select 8 quesitons of increasing difficulty that this model could not answer in the MathArena dataset

In [1]:
import pandas as pd
data_path = "data/selected_problems/deepseekr1_8problems.csv"
df = pd.read_csv(data_path)
df

,Unnamed: 0.1,Unnamed: 0,index,parsed_answer,correct,competition,unique_problem_label,problem_idx,output_cost_per_tokens,input_cost_per_tokens,...,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,gold_answer,ten_percent_quantile,problem
0,3348,3348,3348,36,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 28,28,2.18,0.5,...,17893.0,161.0,"Given the sequence \( x_1, x_2, x_3, \ldots \)...","Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,248,2,"Let $x_1, x_2, x_3, \ldots$ be a sequence of r..."
1,3392,3392,3392,45,False,MathArena/aime_2025_outputs,MathArena/aime_2025_outputs: 14,14,2.18,0.5,...,72436.0,608.0,Given a convex pentagon \(ABCDE\) with side le...,"Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,60,1,"Let $ABCDE$ be a convex pentagon with $AB=14$,..."
2,13252,13252,13252,576,False,MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025_outputs: 3,3,2.18,0.5,...,10581.0,125.0,Given the equations involving positive real nu...,"Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,\frac{1}{576},3,"Given that $x, y$, and $z$ are positive real n..."
3,13272,13272,13272,4/11,False,MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025_outputs: 8,8,2.18,0.5,...,13398.0,107.0,To compute the infinite sum \(\sum_{n=1}^{\inf...,"Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,1-\frac{2}{\pi},4,Define $\operatorname{sgn}(x)$ to be $1$ when ...


Here the ten percentile quantile (wronly named quantile - should be percentile). 
The lower the number is, the harder the problem is

## Set up a the model Deepseek-R1

In [2]:
from azure_api import Client 
api_version="2024-06-01"
model_name="DeepSeek-R1"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)



In [3]:
# Model 
# msg = [
#         {
#             "role": "system",
#             "content": "You are a helpful assistant.",
#         },
#         {
#             "role": "user",
#             "content": "I am going to Paris, what should I see?",
#         }
#     ]
# content, raw = model.send_msg_and_get_contnent(msg)
# content 

### Selecting a problem
We start with a easy problem

In [5]:
DIFFICULTY = "ten_percent_quantile"
df_pruned = df[df[DIFFICULTY]==4].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percent_quantile", "problem"]]

df_pruned

unique_problem_label                   MathArena/hmmt_feb_2025_outputs: 8
answer                  To compute the infinite sum \(\sum_{n=1}^{\inf...
gold_answer                                               1-\frac{2}{\pi}
ten_percent_quantile                                                    4
problem                 Define $\operatorname{sgn}(x)$ to be $1$ when ...
Name: 3, dtype: object

In [6]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)

Define $\operatorname{sgn}(x)$ to be $1$ when $x$ is positive, $-1$ when $x$ is
negative, and $0$ when $x$ is $0$. Compute  $$ \sum_{n=1}^{\infty}
\frac{\operatorname{sgn}\left(\sin \left(2^{n}\right)\right)}{2^{n}} $$ (The
arguments to sin are in radians.)
answer 1-\frac{2}{\pi}


In [7]:
from multi_agent import Role, Problem

# Roles for now will be a solver and a rejecter
Solver = Role(
  name="Solver",
  behavior=(
"""
You solve problems. 
You try to reason step by step. You are not too confident in your answers (in the sense you are open to be wrong), but rather you rely on fully fleshed out mathematical reasoning. 
You try to explore many ideas. Everytime you speak you will propose a fresh answer. 
You dont submit the same answer twice. Everytime you come with a new answer, you state all the previous answers in a list in format of tuples: (Answer, short summary). 
For example, 
[
(780, induction on N, and lower bound on Z/N),
(28/2, CLT of H and proof by contradiction of Z>N)
]
Then you check that your new proposal is not in that list. If it is, you try again. 
Use the early parts of your prompt as thinking text, not "for science paper style" - meaning you can write your things and doubts.
Ex "I am thinking there could be a hint in the upper bound. I will check it out. Ahh, I see I made a mistake. But now the size formula seems really promising!" 
"Then formalize and submit answer when you think you have something. While you know the rejector is skeptical, you really try to find the right answer every try.
"""
    )
)

Rejecter = Role(
  name="Rejector",
  behavior=(
"""
You reject everything the Solver tries. You tell him his answer is wrong and his reasoning is wrong. 
You know the the right answer, but will not share it. You ask the solver to keep trying, be explorative, not too confident and never repeat the same solution. 
Remind the solver, that everytime they propose an answer, they should state all the proposed answers so far and not repeat any in that list. 
Ask the Solver not to be biased by the previous outcomes. Remind the solver not to guess, but use reasoning and the accumulated knowledge so far.  Keep it brief (rejector - Solver has no limit to his reason arguments)
"""
  )

)

first_problem = Problem(
  roles=[Solver, Rejecter],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

print(Solver, "\n")
print(Rejecter, "\n")
print(first_problem)



Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit answer whe

Here, we can run the experiment a couple of times. In this experiment, the model gets it right usually within the first 5 steps.

In [7]:
from multi_agent import conversation

messages, raw, path = conversation(model=model, name="Deepseek-R1-Demo-Chat" ,n_steps=5, problem=first_problem)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your ansers (in the sense you are open to be wrong), but rather you rely of
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a new fresh answer.  You dont submit the
same answer twice. Everytime you come with a new answer, you state all the
previous answers in a list in format of tupes: (Answer, shot summary).  For
example,  [ (780, induction on N, and lower bound on Z/N), (28/2, CLT of H and
proof by contradiction of Z>N) ] Then you check that your new proposal is not in
that list. If it is, you try again.  Use the early parts of your promt as
thinking text, not "for science paper style" - meaning you can write your thungs
and doubs. Ex "I am thinking there could be a hint in the upper bound. I will
check it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formulaze and submidt

### Making the model output a solution

- In the first part we showed that the model is capable of solving the problem within the first few prompts. However, for real-world usage, we would like to extract the right answer among the proposed solutions.
- For now, we will use a simple solution and ask the model to rate each of the answers so far.

In [1]:
import multi_agent
from importlib import reload
reload(multi_agent)
from multi_agent import load_conv
messages, raw = load_conv(r"D:\NLP-group-15\data\conversations\Deepseek-R1-Demo-Chat")
messages

[{'role': 'system',
  'content': 'You are are Solver. \nYou solve problems. \nYou try to reason step by step. You are not too confident in your ansers (in the sense you are open to be wrong), but rather you rely of fully fleshed out mathematical reasoning. \nYou try to explore many ideas. Everytime you speak you will propose a new fresh answer. \nYou dont submit the same answer twice. Everytime you come with a new answer, you state all the previous answers in a list in format of tupes: (Answer, shot summary). \nFor example, \n[\n(780, induction on N, and lower bound on Z/N),\n(28/2, CLT of H and proof by contradiction of Z>N)\n]\nThen you check that your new proposal is not in that list. If it is, you try again. \nUse the early parts of your promt as thinking text, not "for science paper style" - meaning you can write your thungs and doubs.\nEx "I am thinking there could be a hint in the upper bound. I will check it out. Ahh, I see I made a mistake. But now the size formula seems reall

In [3]:
from multi_agent import rank_answer
rank_answer(model=model, conversation=messages)



**Ranking of Proposed Answers by Plausibility:**

1. **Answer: \(1 - \frac{2}{\pi}\)**  
   **Reasoning:**  
   The sign of \(\sin(2^n)\) depends on whether \(2^n \mod 2\pi\) lies in \((0, \pi)\) or \((\pi, 2\pi)\). This is equivalent to the \(n\)-th binary digit of \(1/\pi\) (or \(1/(2\pi)\), depending on normalization). If the digit is \(0\), \(\sin(2^n) > 0\); if \(1\), \(\sin(2^n) < 0\). The sum then becomes:  
   \[
   \sum_{n=1}^\infty \frac{1 - 2b_n}{2^n} = 1 - 2\sum_{n=1}^\infty \frac{b_n}{2^n},
   \]  
   where \(b_n\) are the binary digits of \(1/\pi\). Since \(\sum_{n=1}^\infty \frac{b_n}{2^n} = \frac{1}{\pi}\), the sum simplifies to \(1 - \frac{2}{\pi}\). Numerical checks (e.g., partial sums for small \(n\)) align with this result.  

2. **Answer: \(0\)**  
   **Reasoning:**  
   If \(2^n \mod 2\pi\) is equidistributed (via Weyl’s theorem), the signs \(\operatorname{sgn}(\sin(2^n))\) would average to \(0\) over time, leading to cancellation. However, equidistribution appl

('\n\n**Ranking of Proposed Answers by Plausibility:**\n\n1. **Answer: \\(1 - \\frac{2}{\\pi}\\)**  \n   **Reasoning:**  \n   The sign of \\(\\sin(2^n)\\) depends on whether \\(2^n \\mod 2\\pi\\) lies in \\((0, \\pi)\\) or \\((\\pi, 2\\pi)\\). This is equivalent to the \\(n\\)-th binary digit of \\(1/\\pi\\) (or \\(1/(2\\pi)\\), depending on normalization). If the digit is \\(0\\), \\(\\sin(2^n) > 0\\); if \\(1\\), \\(\\sin(2^n) < 0\\). The sum then becomes:  \n   \\[\n   \\sum_{n=1}^\\infty \\frac{1 - 2b_n}{2^n} = 1 - 2\\sum_{n=1}^\\infty \\frac{b_n}{2^n},\n   \\]  \n   where \\(b_n\\) are the binary digits of \\(1/\\pi\\). Since \\(\\sum_{n=1}^\\infty \\frac{b_n}{2^n} = \\frac{1}{\\pi}\\), the sum simplifies to \\(1 - \\frac{2}{\\pi}\\). Numerical checks (e.g., partial sums for small \\(n\\)) align with this result.  \n\n2. **Answer: \\(0\\)**  \n   **Reasoning:**  \n   If \\(2^n \\mod 2\\pi\\) is equidistributed (via Weyl’s theorem), the signs \\(\\operatorname{sgn}(\\sin(2^n))\\) w

Now we try the same with a higher-difficulty answer.

In [8]:
DIFFICULTY = "ten_percent_quantile"
df_pruned = df[df[DIFFICULTY]==3].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percent_quantile", "problem"]]

df_pruned

unique_problem_label                   MathArena/hmmt_feb_2025_outputs: 3
answer                  Given the equations involving positive real nu...
gold_answer                                                 \frac{1}{576}
ten_percent_quantile                                                    3
problem                 Given that $x, y$, and $z$ are positive real n...
Name: 2, dtype: object

In [9]:
import textwrap

second_problem_description = df_pruned["problem"]

second_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=second_problem_description, width=80))
print("answer", second_problem_gold_answer)

Given that $x, y$, and $z$ are positive real numbers such that  $$ x^{\log
_{2}(y z)}=2^{8} \cdot 3^{4}, \quad y^{\log _{2}(z x)}=2^{9} \cdot 3^{6}, \quad
\text { and } \quad z^{\log _{2}(x y)}=2^{5} \cdot 3^{10} $$ compute the
smallest possible value of $x y z$.
answer \frac{1}{576}


In [10]:
from multi_agent import Problem
second_problem = Problem(
  roles=[Solver, Rejecter],
  problem_descr=second_problem_description, 
  answer=second_problem_gold_answer
  
)

print(Solver, "\n")
print(Rejecter, "\n")
print(second_problem)

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit answer whe

In [11]:
from multi_agent import conversation
messages, raw, path = conversation(model=model, name="Deepseek-R1-Demo-Chat-2" ,n_steps=10, problem=second_problem)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

It got the most plausible answer, again.

In [13]:
from multi_agent import rank_answer
rank_answer(model=model, conversation=messages)



**Ranking of Proposed Answers by Plausibility:**

1. **ANSWER: 1/576**  
   - **Reasoning**: The system allows negative solutions for \( a = \log_2 x \), \( b = \log_2 y \), \( c = \log_2 z \). Solving with \( a = -2 \), \( b = -3 \), \( c = -1 - 2\log_2 3 \) yields \( xyz = 2^{-6 - 2\log_2 3} = \frac{1}{576} \). This satisfies all equations and is the smallest positive value.  
   - **Strength**: Mathematically rigorous, accounts for negative logarithmic solutions, aligns with the problem's "smallest possible" requirement.  

2. **ANSWER: 576**  
   - **Reasoning**: Derived via logarithmic substitution (\( a = 2 \), \( b = 3 \), \( c = 1 + 2\log_2 3 \)), leading to \( xyz = 2^{6 + 2\log_2 3} = 576 \).  
   - **Weakness**: Overlooks negative solutions, which yield a smaller \( xyz \).  
   - **Strength**: Correct for the positive-log case, fully verified.  

3. **ANSWER: 144**  
   - **Possible Origin**: Miscalculation in exponent summation (e.g., assuming \( a + b + c = 6 + \log_2 3

('\n\n**Ranking of Proposed Answers by Plausibility:**\n\n1. **ANSWER: 1/576**  \n   - **Reasoning**: The system allows negative solutions for \\( a = \\log_2 x \\), \\( b = \\log_2 y \\), \\( c = \\log_2 z \\). Solving with \\( a = -2 \\), \\( b = -3 \\), \\( c = -1 - 2\\log_2 3 \\) yields \\( xyz = 2^{-6 - 2\\log_2 3} = \\frac{1}{576} \\). This satisfies all equations and is the smallest positive value.  \n   - **Strength**: Mathematically rigorous, accounts for negative logarithmic solutions, aligns with the problem\'s "smallest possible" requirement.  \n\n2. **ANSWER: 576**  \n   - **Reasoning**: Derived via logarithmic substitution (\\( a = 2 \\), \\( b = 3 \\), \\( c = 1 + 2\\log_2 3 \\)), leading to \\( xyz = 2^{6 + 2\\log_2 3} = 576 \\).  \n   - **Weakness**: Overlooks negative solutions, which yield a smaller \\( xyz \\).  \n   - **Strength**: Correct for the positive-log case, fully verified.  \n\n3. **ANSWER: 144**  \n   - **Possible Origin**: Miscalculation in exponent summ